# 02 — U-Net Training on LandCover.ai

Train a U-Net model for **5-class land cover segmentation** on the [LandCover.ai](https://landcover.ai/) dataset.

Classes: `background (0)`, `building (1)`, `woodland (2)`, `water (3)`, `road (4)`

**Run on Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip install -q torch torchvision albumentations matplotlib

In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Download and Prepare LandCover.ai

In [ ]:
DATA_DIR = Path('landcoverai')
DATA_DIR.mkdir(exist_ok=True)

ZIP_URL = 'https://landcover.ai.linuxpolska.com/download/landcover.ai.v1.zip'
ZIP_PATH = DATA_DIR / 'landcover.ai.v1.zip'

if not (DATA_DIR / 'images').exists():
    print('Downloading LandCover.ai (~1.4 GB)...')
    !wget -q --show-progress -O {ZIP_PATH} {ZIP_URL}
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    ZIP_PATH.unlink()
    print('Done!')
else:
    print('Dataset already downloaded.')

### Run `split.py` to create 512×512 chips

In [ ]:
OUTPUT_DIR = DATA_DIR / 'output'

if not OUTPUT_DIR.exists():
    # LandCover.ai ships split.py inside the zip
    split_script = DATA_DIR / 'split.py'
    if split_script.exists():
        !cd {DATA_DIR} && python split.py
    else:
        # Manual fallback: chip images ourselves
        print('split.py not found — chipping manually...')
        OUTPUT_DIR.mkdir(exist_ok=True)
        (OUTPUT_DIR / 'images').mkdir(exist_ok=True)
        (OUTPUT_DIR / 'masks').mkdir(exist_ok=True)

        TILE = 512
        for img_path in sorted((DATA_DIR / 'images').glob('*.tif')):
            name = img_path.stem
            img = np.array(Image.open(img_path))
            msk = np.array(Image.open(DATA_DIR / 'masks' / img_path.name))
            h, w = img.shape[:2]
            idx = 0
            for y in range(0, h - TILE + 1, TILE):
                for x in range(0, w - TILE + 1, TILE):
                    chip_img = img[y:y+TILE, x:x+TILE]
                    chip_msk = msk[y:y+TILE, x:x+TILE]
                    Image.fromarray(chip_img).save(OUTPUT_DIR / 'images' / f'{name}_{idx}.png')
                    Image.fromarray(chip_msk).save(OUTPUT_DIR / 'masks' / f'{name}_{idx}.png')
                    idx += 1
        print(f'Created {len(list((OUTPUT_DIR / "images").glob("*.png")))} chips.')
else:
    print('Chips already exist.')

# List chip files
chip_image_dir = OUTPUT_DIR / 'images'
chip_mask_dir = OUTPUT_DIR / 'masks'
all_chips = sorted([f.stem for f in chip_image_dir.iterdir() if f.suffix in ('.png', '.tif', '.jpg')])
print(f'Total chips: {len(all_chips)}')

### Train / Val / Test Split

Use the official `train.txt`, `val.txt`, `test.txt` files shipped with LandCover.ai (they list the original large image names). We map chip names back to their parent image.

In [ ]:
def read_split_file(path):
    """Read a split file and return set of base image names."""
    if not path.exists():
        return set()
    with open(path) as f:
        return {line.strip() for line in f if line.strip()}

train_names = read_split_file(DATA_DIR / 'train.txt')
val_names = read_split_file(DATA_DIR / 'val.txt')
test_names = read_split_file(DATA_DIR / 'test.txt')

# If split files are missing, fall back to 70/15/15 random split
if not train_names:
    print('Split files not found — using random 70/15/15 split.')
    rng = np.random.default_rng(42)
    rng.shuffle(all_chips)
    n = len(all_chips)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    train_chips = all_chips[:n_train]
    val_chips = all_chips[n_train:n_train + n_val]
    test_chips = all_chips[n_train + n_val:]
else:
    # Map chips to splits based on parent image name prefix
    def get_parent_name(chip_name):
        # Chip names are like "M-33-20-D-c-4-2_3" → parent is "M-33-20-D-c-4-2"
        parts = chip_name.rsplit('_', 1)
        return parts[0] if len(parts) > 1 else chip_name

    train_chips = [c for c in all_chips if get_parent_name(c) in train_names]
    val_chips = [c for c in all_chips if get_parent_name(c) in val_names]
    test_chips = [c for c in all_chips if get_parent_name(c) in test_names]

print(f'Train: {len(train_chips)}, Val: {len(val_chips)}, Test: {len(test_chips)}')

## 2. Dataset & DataLoader

In [ ]:
IMG_SIZE = 256
NUM_CLASSES = 5

class LandCoverDataset(Dataset):
    def __init__(self, chip_names, image_dir, mask_dir, img_size=256):
        self.chip_names = chip_names
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size

        # Detect file extension
        sample = chip_names[0]
        for ext in ['.png', '.tif', '.jpg']:
            if (self.image_dir / f'{sample}{ext}').exists():
                self.ext = ext
                break
        else:
            self.ext = '.png'

    def __len__(self):
        return len(self.chip_names)

    def __getitem__(self, idx):
        name = self.chip_names[idx]
        img = Image.open(self.image_dir / f'{name}{self.ext}').convert('RGB')
        msk = Image.open(self.mask_dir / f'{name}{self.ext}')

        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        msk = msk.resize((self.img_size, self.img_size), Image.NEAREST)

        img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        msk = torch.from_numpy(np.array(msk)).long()
        # Clamp mask values to valid range
        msk = msk.clamp(0, NUM_CLASSES - 1)

        return img, msk

train_ds = LandCoverDataset(train_chips, chip_image_dir, chip_mask_dir, IMG_SIZE)
val_ds = LandCoverDataset(val_chips, chip_image_dir, chip_mask_dir, IMG_SIZE)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

# Sanity check
imgs, msks = next(iter(train_loader))
print(f'Image batch: {imgs.shape}, Mask batch: {msks.shape}')
print(f'Mask unique values: {torch.unique(msks).tolist()}')

## 3. U-Net Model (5 classes)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=5):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.bottleneck = DoubleConv(512, 1024)
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.final = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.final(d1)

model = UNet(in_ch=3, num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'U-Net parameters: {total_params:,}')

## 4. Loss Function (CrossEntropy + Dice)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        target_oh = torch.zeros_like(pred)
        target_oh.scatter_(1, target.unsqueeze(1), 1)
        intersection = (pred * target_oh).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target_oh.sum(dim=(2, 3))
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice = DiceLoss()
        self.ce = nn.CrossEntropyLoss()
        self.dw = dice_weight
        self.cw = ce_weight

    def forward(self, pred, target):
        return self.dw * self.dice(pred, target) + self.cw * self.ce(pred, target)

criterion = CombinedLoss()
print('Loss: CrossEntropy + Dice (equal weight)')

## 5. Training Loop with Validation

In [ ]:
EPOCHS = 25
LR = 1e-3

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

def compute_iou(pred_mask, true_mask, num_classes):
    """Compute mean IoU."""
    ious = []
    for c in range(num_classes):
        pred_c = (pred_mask == c)
        true_c = (true_mask == c)
        intersection = (pred_c & true_c).sum().item()
        union = (pred_c | true_c).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0

history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_acc': []}
best_iou = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss = 0
    val_iou = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()
            preds = outputs.argmax(dim=1)
            val_iou += compute_iou(preds.cpu(), masks.cpu(), NUM_CLASSES)
            val_correct += (preds == masks).sum().item()
            val_total += masks.numel()

    val_loss /= len(val_loader)
    val_iou /= len(val_loader)
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_acc'].append(val_acc)

    scheduler.step(val_iou)

    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'unet_sentinel2.pth')

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Val IoU: {val_iou:.4f} | Val Acc: {val_acc:.4f} | LR: {lr_now:.1e}')

print(f'\nBest Val IoU: {best_iou:.4f}')
print('Best weights saved to unet_sentinel2.pth')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['val_iou'], color='green')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Validation Mean IoU')

axes[2].plot(history['val_acc'], color='orange')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
axes[2].set_title('Validation Pixel Accuracy')

plt.tight_layout()
plt.show()

## 7. Visualize Predictions vs Ground Truth

In [ ]:
CLASS_NAMES = {0: 'Background', 1: 'Building', 2: 'Woodland', 3: 'Water', 4: 'Road'}
CLASS_COLORS = {
    0: [0, 0, 0], 1: [255, 0, 0], 2: [0, 180, 0], 3: [0, 0, 255], 4: [255, 255, 0]
}

def colorize_mask(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id, color in CLASS_COLORS.items():
        rgb[mask == cls_id] = color
    return rgb

# Load best weights
model.load_state_dict(torch.load('unet_sentinel2.pth', map_location=device, weights_only=True))
model.eval()

# Get a batch from validation
images, masks = next(iter(val_loader))
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu().numpy()

n_show = min(4, len(images))
fig, axes = plt.subplots(n_show, 3, figsize=(15, 5 * n_show))
for i in range(n_show):
    img_np = images[i].permute(1, 2, 0).numpy()
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title('Input')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(colorize_mask(masks[i].numpy()))
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(colorize_mask(preds[i]))
    axes[i, 2].set_title('Prediction')
    axes[i, 2].axis('off')

plt.suptitle('Predictions vs Ground Truth', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Download weights from Colab
print('Weights saved as unet_sentinel2.pth')
print(f'File size: {os.path.getsize("unet_sentinel2.pth") / 1e6:.1f} MB')

# Uncomment to download:
# from google.colab import files
# files.download('unet_sentinel2.pth')